In [ ]:
import torch
import time
import datetime
import random
import numpy as np
import polars as pl
from scipy.sparse import csr_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import train_test_split
import seaborn as sns

In [ ]:
EPOCHS = 10
BATCH_SIZE = 100

In [ ]:
torch.set_default_dtype(torch.float32)
if torch.cuda.is_available():
    torch.set_default_device(0)
    print("Running on the GPU")
else:
    print("Running on the CPU")

[bood_id_map.csv](https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/book_id_map.csv)
[user_id_map.csv](https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/user_id_map.csv)
[goodreads_books.json.gz](https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_books.json.gz)
[goodreads_reviews_dedup.json.gz](https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_reviews_dedup.json.gz)
[goodreads_interactions.csv](https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_interactions.csv)
[goodreads_interactions_dedup.json.gz](https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_interactions_dedup.json.gz)

In [ ]:
# book_ids = pl.read_csv('./data/book_id_map.csv')
# user_ids = pl.read_csv('./data/user_id_map.csv')
# books = pl.read_ndjson('./data/goodreads_books.json')
# reviews = pl.read_ndjson('./data/goodreads_reviews_dedup.json')
# interactions = pl.read_csv('./data/goodreads_interactions.csv')
# interactions_dedup = pl.read_ndjson('./data/goodreads_interactions_dedup.json')

# book_ids.write_ipc('./data/book_id_map.feather')
# user_ids.write_ipc('./data/user_id_map.feather')
# books.write_ipc('./data/goodreads_books.feather')
# reviews.write_ipc('./data/goodreads_reviews_dedup.feather')
# interactions.write_ipc('./data/goodreads_interactions.feather')
# interactions_dedup.write_ipc('./data/goodreads_interactions_dedup.feather')

In [ ]:
# book_ids = pl.read_ipc('./data/book_id_map.feather')
# user_ids = pl.read_ipc('./data/user_id_map.feather')
books = pl.read_ipc('./data/goodreads_books.feather')
# reviews = pl.read_ipc('./data/goodreads_reviews_dedup.feather')
interactions = pl.read_ipc('./data/goodreads_interactions.feather')
# interactions_dedup = pl.read_ipc('./data/goodreads_interactions_dedup.feather')

In [ ]:
# book_ids.head()

In [ ]:
# book_ids.collect_schema()

In [ ]:
# user_ids.head()

In [ ]:
# user_ids.collect_schema()

In [ ]:
books.head()

- ~~('isbn', String)~~
- ('text_reviews_count', String) -> Integer
- ~~('series', List(String))~~
- ('country_code', String) -> Boolean
- ('language_code', String) -> One Hot Encoding
- ~~('popular_shelves', List(Struct({'count': String, 'name': String})))~~
- ~~('asin', String)~~
- ('is_ebook', String) -> Boolean
- ('average_rating', String) -> Float
- ~~('kindle_asin', String)~~
- ~~('similar_books', List(String))~~
- ~~('description', String)~~
- ('format', String) -> One Hot Encoding
- ~~('link', String)~~
- ~~('authors', List(Struct({'author_id': String, 'role': String})))~~
- ~~('publisher', String)~~
- ('num_pages', String) -> Integer / Float
- ~~('publication_day', String)~~
- ~~('isbn13', String)~~
- ~~('publication_month', String)~~
- ~~('edition_information', String)~~
- ('publication_year', String) -> Integer / Float
- ~~('url', String)~~
- ~~('image_url', String)~~
- ~~('book_id', String)~~
- ('ratings_count', String) -> Integer / Float
- ~~('work_id', String)~~
- ~~('title', String)~~
- ~~('title_without_series', String)~~

In [ ]:
books.drop_in_place('isbn')
books.drop_in_place('series')
books.drop_in_place('popular_shelves')
books.drop_in_place('asin')
books.drop_in_place('kindle_asin')
books.drop_in_place('similar_books')
books.drop_in_place('description')
books.drop_in_place('link')
books.drop_in_place('authors')
books.drop_in_place('publisher')
books.drop_in_place('isbn13')
books.drop_in_place('edition_information')
books.drop_in_place('url')
books.drop_in_place('image_url')
books.drop_in_place('work_id')
books.drop_in_place('title')
books.drop_in_place('title_without_series')
books.drop_in_place('publication_month')
books.drop_in_place('publication_day')

In [ ]:
books = books.with_columns(pl.col('book_id').cast(pl.Int64))
books = books.filter(pl.col('publication_year').ne(''))
books = books.filter(pl.col('publication_year').cast(pl.Int64).le(2026))

In [ ]:
# books = books.lazy().with_columns(
#     pl.when(pl.col('publication_month').str.len_chars().eq(0)).then(pl.col('publication_month').str.join('1')).otherwise(pl.col('publication_month')).alias('publication_month')
# ).collect(engine='streaming')

In [ ]:
# books = books.lazy().with_columns(
#     pl.when(pl.col('publication_day').str.len_chars().eq(0)).then(pl.col('publication_day').str.join('1')).otherwise(pl.col('publication_day')).alias('publication_day')
# ).collect(engine='streaming')

In [ ]:
# books = books.lazy().with_columns(
#     pl.when(pl.col('publication_year').str.starts_with('200') & pl.col('publication_year').str.len_chars().eq(5)).then(pl.col('publication_year').cast(pl.Int64).sub(18000).cast(pl.String)).otherwise(pl.col('publication_year')).alias('publication_year') # 12 fixes
# ).collect(engine='streaming')

In [ ]:
# books = books.lazy().with_columns(
#     pl.when((pl.col('publication_year').str.starts_with('201') | pl.col('publication_year').str.starts_with('19')) & pl.col('publication_year').str.len_chars().eq(5)).then(pl.col('publication_year').str.slice(0,4)).otherwise(pl.col('publication_year')).alias('publication_year') # 42 fixes
# ).collect(engine='streaming')

In [ ]:
# books = books.lazy().with_columns(
#     pl.when(pl.col('publication_year').str.starts_with('210') & pl.col('publication_year').str.len_chars().eq(5)).then(pl.col('publication_year').cast(pl.Int64).sub(19000).cast(pl.String)).otherwise(pl.col('publication_year')).alias('publication_year') # 5 fixes
# ).collect(engine='streaming')

In [ ]:
# books = books.lazy().with_columns(
#     pl.when(pl.col('publication_year').str.starts_with('220') & pl.col('publication_year').str.len_chars().eq(5)).then(pl.col('publication_year').cast(pl.Int64).sub(20000).cast(pl.String)).otherwise(pl.col('publication_year')).alias('publication_year') # 6 fixes
# ).collect(engine='streaming')

In [ ]:
# books = books.lazy().with_columns(
#     pl.when(pl.col('publication_year').str.starts_with('210') & pl.col('publication_year').str.len_chars().eq(4)).then(pl.col('publication_year').cast(pl.Int64).sub(90).cast(pl.String)).otherwise(pl.col('publication_year')).alias('publication_year') # 8 fixes
# ).collect(engine='streaming')

In [ ]:
# books = books.lazy().with_columns(
#     pl.when(pl.col('publication_year').str.starts_with('30') & pl.col('publication_year').str.len_chars().eq(4)).then(pl.col('publication_year').cast(pl.Int64).sub(1000).cast(pl.String)).otherwise(pl.col('publication_year')).alias('publication_year') # 8 fixes
# ).collect(engine='streaming')

In [ ]:
# books = books.lazy().with_columns(
#     pl.when(pl.col('publication_year').str.starts_with('40') & pl.col('publication_year').str.len_chars().eq(4)).then(pl.col('publication_year').cast(pl.Int64).sub(2000).cast(pl.String)).otherwise(pl.col('publication_year')).alias('publication_year') # 3 fixes
# ).collect(engine='streaming')

In [ ]:
# books = books.lazy().with_columns(
#     pl.when((pl.col('publication_month').cast(pl.Int64).eq(2) | pl.col('publication_month').cast(pl.Int64).eq(4) | pl.col('publication_month').cast(pl.Int64).eq(6) | pl.col('publication_month').cast(pl.Int64).eq(9) | pl.col('publication_month').cast(pl.Int64).eq(11)) & pl.col('publication_day').cast(pl.Int64).ge(29)).then(pl.col('publication_day').str.join('28').slice(-2)).otherwise(pl.col('publication_day')).alias('publication_day')
# ).collect(engine='streaming')

In [ ]:
# books = books.filter(pl.col('publication_year').cast(pl.Int64).le(2026))

In [ ]:
# books = books.with_columns(publication_date=(
#                             pl.when(pl.col('publication_year').eq('')).then(1).otherwise(pl.col('publication_year')) + '-' + 
#                             pl.when(pl.col('publication_month').eq('')).then(1).otherwise(pl.col('publication_month')) + '-' +
#                             pl.when(pl.col('publication_day').eq('')).then(1).otherwise(pl.col('publication_day'))).str.to_date('%Y-%m-%d'))
# books.drop_in_place('publication_year')
# books.drop_in_place('publication_month')
# books.drop_in_place('publication_day')
# books.head()

In [ ]:
# print(f'# Formats: {len(set(books['format']))}')
# print(f'Formats: {set(books['format'])}')
# print(f'# Publishers: {len(set(books['publisher']))}')
# # print(f'Publishers: {set(books['publisher'])}')
# print(f'# Edition Informations: {len(set(books['edition_information']))}')
# # print(f'Edition Informations: {set(books['edition_information'])}')
# print(f'# Language Codes: {len(set(books['language_code']))}')
# print(f'Language Codes: {set(books['language_code'])}')
# print(f'# Country Codes: {len(set(books['country_code']))}')
# print(f'Country Codes: {set(books['country_code'])}')

In [ ]:
books.collect_schema()

In [ ]:
# reviews.head()

In [ ]:
# reviews.collect_schema()

In [ ]:
interactions.head()

In [ ]:
interactions.collect_schema()

In [ ]:
# interactions_dedup.head()

In [ ]:
# print(f'Ratings: {set(interactions_dedup['rating'])}')

In [ ]:
# interactions_dedup.collect_schema()

In [ ]:
interactions = interactions.filter(pl.col('is_read').eq(1))
interactions = interactions.drop('is_read')
interactions = interactions.filter(pl.col('user_id').is_duplicated())

In [ ]:
print(len(interactions))

In [ ]:
print(len(books))

In [ ]:
books.collect_schema()

In [ ]:
books['text_reviews_count'].fill_null(0)

In [ ]:
books = books.with_columns(pl.col('text_reviews_count').str.replace('', '0').cast(pl.Float32))
books = books.with_columns(pl.col('is_ebook').replace('false', '0').replace('true', '1').cast(pl.Float32))
books = books.with_columns(pl.col('average_rating').str.replace('', '0').cast(pl.Float32))
books = books.with_columns(pl.col('num_pages').str.replace('', '0').cast(pl.Float32))
books = books.with_columns(pl.col('publication_year').cast(pl.Float32))
books = books.with_columns(pl.col('ratings_count').replace('', '0').cast(pl.Float32))
books = books.with_columns(pl.col('country_code').replace('', '0').replace('US', '1').cast(pl.Float32))
books = books.to_dummies('language_code')
books = books.to_dummies('format')

In [ ]:
users = books.join(interactions, on='book_id')

In [ ]:
users.group_by('user_id').agg(pl.exclude('book_id').mean())

In [ ]:
print(len(users.columns))

In [ ]:
def normalize_ids(df: pl.DataFrame, id: str):
    uniq_ids = set(df[id].unique())
    ids_to_indices = { orig_id : new_id for new_id, orig_id in enumerate(uniq_ids) }
    new_ids = df.with_columns(pl.col(id).replace(ids_to_indices).cast(pl.Int64))
    indices_to_ids = { new_id : orig_id for orig_id, new_id in ids_to_indices.items()}
    return new_ids, indices_to_ids

In [ ]:
users, _ = normalize_ids(users, 'user_id')

users_n_books = users['book_id'].n_unique()

In [ ]:
gss = GroupShuffleSplit(n_splits=5)

users_train_idx, users_test_idx = next(gss.split(X=users, groups=users['user_id']))
users_train: pl.Dataframe = users[users_train_idx]
users_test:  pl.Dataframe = users[users_test_idx]

In [ ]:
users_test = users_test.filter(pl.col('user_id').is_duplicated())
users_train = users_train.filter(pl.col('user_id').is_in(users_test['user_id'].implode()))

In [ ]:
users_train, users_train_map_id = normalize_ids(users_train, 'user_id')
users_test,  users_test_map_id  = normalize_ids(users_test, 'user_id')

# users_train_csr = csr_matrix((users_train['rating'], (users_train['user_id'], users_train['book_id'])), shape=(users_train['user_id'].n_unique(), users_n_books))

In [ ]:
users_test_seen,   users_test_unseen         = train_test_split(users_test, train_size=0.7, stratify=users_test['user_id'])

users_test_seen,   users_test_seen_map_id    = normalize_ids(users_test_seen,   'user_id')
users_test_unseen, users_test_unseen_map_id  = normalize_ids(users_test_unseen, 'user_id')

# users_test_seen_csr = csr_matrix((users_test_seen['rating'], (users_test_seen['user_id'], users_test_seen['book_id'])), shape=(users_test_seen['user_id'].n_unique(), users_n_books))

In [ ]:
users_train = users_train.group_by('user_id')
users_test_seen = users_test_seen.group_by('user_id').agg(pl.exclude('book_id').mean())
users_test_unseen = users_test_unseen.group_by('user_id').agg(pl.exclude('book_id').mean())

In [ ]:
# train_crow_idx = interactions_train_csr.tocoo().coords[0]
# np.append(train_crow_idx, interactions_train_csr.getnnz(axis=0))
# train_col_idx = interactions_train_csr.tocoo().coords[1]
# np.append(train_crow_idx, interactions_train_csr.getnnz(axis=1))
# train_values = interactions_train_csr.tocoo().data

# test_crow_idx = interactions_test_seen_csr.tocoo().coords[0]
# test_col_idx = interactions_test_seen_csr.tocoo().coords[1]
# test_values = interactions_test_seen_csr.tocoo().data

# interactions_features = torch.sparse_csr_tensor(crow_indices=torch.tensor(train_crow_idx, dtype=torch.int64),
#                                                 col_indices=torch.tensor(train_col_idx, dtype=torch.int64),
#                                                 values=torch.tensor(train_values, dtype=torch.int64))
# interactions_labels = torch.sparse_csr_tensor(crow_indices=torch.tensor(test_crow_idx, dtype=torch.int64),
#                                               col_indices=torch.tensor(test_col_idx, dtype=torch.int64),
#                                               values=torch.tensor(test_values, dtype=torch.int64))

In [ ]:
class Timer(object):
    def __init__(self, name=None, filename=None):
        self.name = name
        self.filename = filename

    def __enter__(self):
        self.tstart = time.time()

    def __exit__(self, type, value, traceback):
        message = 'Elapsed: %.2f seconds' % (time.time() - self.tstart)
        if self.name:
            message = '[%s] ' % self.name + message
        print(message)
        if self.filename:
            with open(self.filename,'a') as file:
                print(str(datetime.datetime.now())+": ",message,file=file)

In [ ]:
# print(interactions['user_id'].n_unique())
# print(interactions_n_books)
# print(interactions_train_csr.nnz)

In [ ]:
class AE(torch.nn.Module):
    def __init__(self):
        super(AE, self).__init__()
        self.model = torch.nn.Sequential()
        self.model.append(torch.nn.Linear(2300,1150))
        self.model.append(torch.nn.ReLU())
        self.model.append(torch.nn.Linear(1150,1150))
        self.model.append(torch.nn.ReLU())
        self.model.append(torch.nn.Linear(1150,2300))
        self.model.append(torch.nn.Sigmoid())
        
    def forward(self, x):
        return self.model(x)

In [ ]:
#loader = torch.utils.data.DataLoader(dataset=interactions_train_csr.data, batch_size=BATCH_SIZE, shuffle=True)
auto_encoder = AE()
loss_function = torch.nn.MSELoss()
optimizer = torch.optim.Adam(auto_encoder.parameters(), lr=0.001, weight_decay=0.00000001)

In [ ]:
training_loss = []

In [ ]:
with Timer('Total time'):
    with Timer('Training time'):
        for epoch in range(EPOCHS):
            for name, data in users_train:
                x = data.drop('user_id').drop('book_id').to_torch(dtype=pl.Float32)
                y = users_test_seen.filter(pl.col('user_id').eq(name[0])).drop('user_id').to_torch(dtype=pl.Float32)
                pred = auto_encoder(x).mean(dim=0, keepdim=True)
                loss = loss_function(pred, y)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                training_loss.append(loss.item())

In [ ]:
sns.lineplot(training_loss)